# Build a simple RNN

## Load the Dataframe

In [1]:
import torch
import torch.nn as nn

if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [2]:
import pandas as pd

In [3]:
from google.colab import drive

drive.flush_and_unmount()
drive.mount("/content/drive", force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [4]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/CFM_2024_Challenge")

print(DATA_DIR.exists())

True


In [5]:
path = Path("/content/drive/MyDrive/CFM_2024_Challenge/Xtrain.parquet")

df = pd.read_parquet(path)

In [ ]:
df[:10]

,obs_id,venue,order_id,action,side,price,bid,ask,bid_size,ask_size,trade,flux
0,0,4,0,A,A,0.30,0.0,0.01,100,1,False,100
1,0,4,1,A,B,-0.17,0.0,0.01,100,1,False,100
2,0,4,2,D,A,0.28,0.0,0.01,100,1,False,-100
3,0,4,3,A,A,0.30,0.0,0.01,100,1,False,100
4,0,4,4,D,A,0.37,0.0,0.01,100,1,False,-100
5,0,1,5,D,A,0.05,0.0,0.01,100,1,False,-1
6,0,4,6,A,B,-0.19,0.0,0.01,100,1,False,100
7,0,4,1,D,B,-0.17,0.0,0.01,100,1,False,-100
8,0,4,3,D,A,0.30,0.0,0.01,100,1,False,-100
9,0,4,7,A,B,-0.02,0.0,0.01,100,1,False,1


## Categorical Features
Version 1: categorical features are one-hot encoded

In [6]:
import numpy as np

In [7]:
# For the moment, we will transform the following coordinates
# as one-hot
cat_columns = ["venue", "action", "side", "trade"]

for col in cat_columns:
    print(col, df[col].unique())

venue [4 1 5 2 0 3]
action ['A' 'D' 'U']
side ['A' 'B']
trade [False  True]


In [8]:
# get_dummies transforms the selected columns into one-hot
X = pd.get_dummies(
    df,
    columns=cat_columns,
    dtype=np.float32
)

In [ ]:
X.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080000 entries, 0 to 16079999
Data columns (total 21 columns):
 #   Column       Dtype  
---  ------       -----  
 0   obs_id       int64  
 1   order_id     int64  
 2   price        float64
 3   bid          float64
 4   ask          float64
 5   bid_size     int64  
 6   ask_size     int64  
 7   flux         int64  
 8   venue_0      float32
 9   venue_1      float32
 10  venue_2      float32
 11  venue_3      float32
 12  venue_4      float32
 13  venue_5      float32
 14  action_A     float32
 15  action_D     float32
 16  action_U     float32
 17  side_A       float32
 18  side_B       float32
 19  trade_False  float32
 20  trade_True   float32
dtypes: float32(13), float64(3), int64(5)
memory usage: 1.7 GB


In [9]:
# we remove obs_id and order_id for the moment
X.drop(columns=["obs_id", "order_id"], inplace=True)

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080000 entries, 0 to 16079999
Data columns (total 19 columns):
 #   Column       Dtype  
---  ------       -----  
 0   price        float64
 1   bid          float64
 2   ask          float64
 3   bid_size     int64  
 4   ask_size     int64  
 5   flux         int64  
 6   venue_0      float32
 7   venue_1      float32
 8   venue_2      float32
 9   venue_3      float32
 10  venue_4      float32
 11  venue_5      float32
 12  action_A     float32
 13  action_D     float32
 14  action_U     float32
 15  side_A       float32
 16  side_B       float32
 17  trade_False  float32
 18  trade_True   float32
dtypes: float32(13), float64(3), int64(3)
memory usage: 1.5 GB


In [10]:
## let's reduce the size of X

# Continuous variables: float64 to float32
float_cols = ["price", "bid", "ask"]
X[float_cols] = X[float_cols].astype("float32")

# Integer-valued features: use smaller integers
X["bid_size"] = X["bid_size"].astype("int32")
X["ask_size"] = X["ask_size"].astype("int32")
X["flux"] = X["flux"].astype("int32")

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080000 entries, 0 to 16079999
Data columns (total 19 columns):
 #   Column       Dtype  
---  ------       -----  
 0   price        float32
 1   bid          float32
 2   ask          float32
 3   bid_size     int32  
 4   ask_size     int32  
 5   flux         int32  
 6   venue_0      float32
 7   venue_1      float32
 8   venue_2      float32
 9   venue_3      float32
 10  venue_4      float32
 11  venue_5      float32
 12  action_A     float32
 13  action_D     float32
 14  action_U     float32
 15  side_A       float32
 16  side_B       float32
 17  trade_False  float32
 18  trade_True   float32
dtypes: float32(16), int32(3)
memory usage: 1.1 GB


## Labels

In [11]:
y_df = pd.read_csv("/content/drive/MyDrive/CFM_2024_Challenge/y_train.csv")
y = torch.tensor(y_df["eqt_code_cat"].to_numpy(), dtype=torch.long)
del y_df

## Normalization
We want to split the data into training and validatio, and normalize the validation set according to the training statistics.

In [12]:
num_cols = ["price", "bid", "ask", "bid_size", "ask_size", "flux"]

n = len(y)

perm = torch.randperm(n) #random indices from 0 to n

train_size = int(0.8 * n)

# one index correspond to one observation
train_idx = perm[:train_size].numpy()
val_idx = perm[train_size:].numpy()

In [13]:
# convert the sample index to the row index in X
# e.g. 2-->200, .... 299

train_rows = np.concatenate([np.arange(i * 100, (i + 1) * 100) for i in train_idx])

In [14]:
## Statistics only from the TRAINING data:

means = X.iloc[train_rows][num_cols].mean()
stds = X.iloc[train_rows][num_cols].std()

In [15]:
# normalize everything using the training data

X_scaled = X.copy()


X_scaled[num_cols] = (
    X_scaled[num_cols] - means
) / stds


In [16]:
# convert X_scaled to a tensor
X_tensor = torch.tensor(X_scaled.to_numpy(), dtype=torch.float32)

### Free the memory

In [17]:
del X, X_scaled # we don't need the pd dataframe X anymore

import gc # garbage collector

gc.collect()


81

## Dataset Class

In [18]:
from torch.utils.data import Dataset, DataLoader

class StockDataset(Dataset):
    def __init__(self, X, y, indices): #indices will be e.g. train_idx (1 index per observation)
      # convert the dataset to a tensor
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices) # e.g. how many indices in train_idx

    # when the dataloader runs the loop i=0,...36
    # this goes through self.indices[i] so that it takes only inidces in train_idx
    def __getitem__(self, i):
        index = self.indices[i]

        start = index * 100
        end = start + 100

        return self.X[start:end], self.y[index]

## DataLoaders

In [19]:
batch_size = 256

train_set = StockDataset(X_tensor, y, train_idx)
val_set   = StockDataset(X_tensor, y, val_idx)

del X_tensor, y # we do not need these in menory anymore
gc.collect()

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

In [ ]:
train_set[0][0].shape

torch.Size([100, 19])

## Training Function

In [20]:
%pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 51.8 MB/s eta 0:00:00


In [21]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None,
          early_stopping=5):  # ADDED
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}

    best_val = -float("inf")  # ADDED
    best_state = None  # ADDED
    epochs_without_improvement = 0  # ADDED

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)

        # ADDED
        if val_metric > best_val:
            best_val = val_metric
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
            best_string = " Best!"
        else:
            epochs_without_improvement += 1
            best_string = ""

        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}"
              f"{best_string}")  # ADDED best_string

        # ADDED
        if epochs_without_improvement >= early_stopping:
            print(f"Early stopping at epoch {epoch + 1}. "
                  f"Best validation metric: {best_val:.2%}")
            break

    model.load_state_dict(best_state)  # ADDED

    return history

# Bidirectional GRU

In [ ]:
class StockModelGruBi(nn.Module):
    def __init__(self, n_layers=2, input_dim=19,
                 hidden_dim=64, dropout=0.2):
        super().__init__()

        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=n_layers,
            batch_first=True, dropout=dropout, bidirectional=True)

        self.output = nn.Linear(2 * hidden_dim, 24)  # the output dimension of the GRU layer is now doubles

    def forward(self, X):
      _outputs, hidden_states = self.gru(X)

      # concatenate forward and backward hidden states
      n_dims = self.output.in_features
      top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)

      return self.output(top_states)

In [ ]:
torch.manual_seed(42)
model = StockModelGruBi()
model.to(device)

n_epochs = 40
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=24).to(device)

history = train(model, optimizer, xentropy, accuracy, train_loader, val_loader,
                n_epochs)

Epoch 1/40,                      train loss: 2.4439, train metric: 20.92%, valid metric: 26.72%
Epoch 2/40,                      train loss: 2.0739, train metric: 32.53%, valid metric: 36.58%
Epoch 3/40,                      train loss: 1.8774, train metric: 38.54%, valid metric: 41.66%
Epoch 4/40,                      train loss: 1.7353, train metric: 43.04%, valid metric: 43.38%
Epoch 5/40,                      train loss: 1.6252, train metric: 46.37%, valid metric: 46.84%
Epoch 6/40,                      train loss: 1.5351, train metric: 49.10%, valid metric: 49.80%
Epoch 7/40,                      train loss: 1.4600, train metric: 51.50%, valid metric: 50.85%
Epoch 8/40,                      train loss: 1.3962, train metric: 53.32%, valid metric: 51.85%
Epoch 9/40,                      train loss: 1.3377, train metric: 55.21%, valid metric: 54.67%
Epoch 10/40,                      train loss: 1.2846, train metric: 56.78%, valid metric: 55.73%
Epoch 11/40,                      train

In [ ]:
#history2 = train(model, optimizer, xentropy, accuracy, train_loader, val_loader,n_epochs=10)

#Inception

In [26]:
from functools import partial
import torch.nn.functional as F

## Inception 1
Proof of concept: how good is an iception module good at classifiying stock? The idea is to put it in paralled to a RNN module.

In [ ]:

# n1 n2 n3 n4
#    n5 n6 n7
class InceptionModule(nn.Module):
  def __init__(self, in_channels, n1, n2, n3, n4, n5, n6, n7):
    super().__init__()
    Conv1d = partial(nn.Conv1d,
                            stride=1, padding="same", bias=False
                            )
    DefaultMaxpool = partial(nn.MaxPool1d,
                             stride=1, padding=n7//2)
    self.column1 = Conv1d(in_channels = in_channels,
                                 out_channels = in_channels//4,
                                 kernel_size=n1)
    self.column2 = nn.Sequential(
        Conv1d(in_channels = in_channels,
                      out_channels = in_channels//4,
                      kernel_size=n5),
        nn.ReLU(),
        Conv1d(in_channels = in_channels//4,
                      out_channels = in_channels//4,
                      kernel_size=n2)
    )
    self.column3 = nn.Sequential(
        Conv1d(in_channels = in_channels,
                      out_channels = in_channels//4,
                      kernel_size=n6),
        nn.ReLU(),
        Conv1d(in_channels = in_channels//4,
                      out_channels = in_channels//4,
                      kernel_size=n3)
    )
    self.column4 = nn.Sequential(
        DefaultMaxpool(kernel_size=n7),
        Conv1d(in_channels = in_channels,
                      out_channels = in_channels//4,
                      kernel_size=n4)
    )
    self.output_dim = in_channels//4 + in_channels//4 + in_channels//4 + in_channels//4
    self.linear_output = nn.Linear(self.output_dim, 24) #warning, see below to get to this layer we need to pool

  def forward(self, inputs):
    inputs = inputs.permute(0, 2, 1) #[batch, 100, 19] to [batch, 19, 100] 19 channels!

    out1 = self.column1(inputs)
    out2 = self.column2(inputs)
    out3 = self.column3(inputs)
    out4 = self.column4(inputs)

    combined = torch.cat((out1, out2, out3, out4), dim=1)

    inception_output = F.relu(combined)
    #inception_output.shape = (256=batch, 16, 100)

    pooled = inception_output.mean(dim=2)
    #pooled.shape = (256=batch, 16)
    return self.linear_output(pooled)

In [ ]:
torch.manual_seed(42)
model_inception = InceptionModule(19, 1, 3, 5, 1, 1, 1, 3)
model_inception.to(device)

n_epochs = 40
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model_inception.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=24).to(device)

history = train(model_inception, optimizer, xentropy, accuracy, train_loader, val_loader,
                n_epochs)

Epoch 1/40,                      train loss: 2.7595, train metric: 12.70%, valid metric: 17.37%
Epoch 2/40,                      train loss: 2.5397, train metric: 19.25%, valid metric: 20.06%
Epoch 3/40,                      train loss: 2.4685, train metric: 21.17%, valid metric: 21.67%
Epoch 4/40,                      train loss: 2.4276, train metric: 22.48%, valid metric: 23.26%
Epoch 5/40,                      train loss: 2.3975, train metric: 23.34%, valid metric: 23.73%
Epoch 6/40,                      train loss: 2.3748, train metric: 23.90%, valid metric: 23.90%
Epoch 7/40,                      train loss: 2.3578, train metric: 24.42%, valid metric: 24.43%
Epoch 8/40,                      train loss: 2.3442, train metric: 24.81%, valid metric: 25.03%
Epoch 9/40,                      train loss: 2.3333, train metric: 25.20%, valid metric: 25.07%
Epoch 10/40,                      train loss: 2.3242, train metric: 25.36%, valid metric: 25.25%
Epoch 11/40,                      train

In [ ]:
history2 = train(model_inception, optimizer, xentropy, accuracy, train_loader, val_loader,
                40)

Epoch 1/40,                      train loss: 2.2017, train metric: 29.43%, valid metric: 29.28%
Epoch 2/40,                      train loss: 2.2010, train metric: 29.52%, valid metric: 29.21%
Epoch 3/40,                      train loss: 2.1999, train metric: 29.48%, valid metric: 29.38%
Epoch 4/40,                      train loss: 2.1993, train metric: 29.53%, valid metric: 29.56%
Epoch 5/40,                      train loss: 2.1982, train metric: 29.52%, valid metric: 29.56%
Epoch 6/40,                      train loss: 2.1974, train metric: 29.60%, valid metric: 29.63%
Epoch 7/40,                      train loss: 2.1963, train metric: 29.56%, valid metric: 29.68%
Epoch 8/40,                      train loss: 2.1952, train metric: 29.65%, valid metric: 29.38%
Epoch 9/40,                      train loss: 2.1946, train metric: 29.73%, valid metric: 29.43%
Epoch 10/40,                      train loss: 2.1934, train metric: 29.70%, valid metric: 29.73%
Epoch 11/40,                      train

In [ ]:
torch.save(
    model_inception.state_dict(),
    "/content/drive/MyDrive/CFM_2024_Challenge/inception_scaled_1.pt"
)

## Inception 2
Similar to inception 1 but we increase the channel output of the branches from 4 to 16. We stack 2 linear layers on top.

In [ ]:
# increase the number of output channels from the single branches
# n1 n2 n3 n4
#    n5 n6 n7
class InceptionModule2(nn.Module):
  def __init__(self, in_channels, n1, n2, n3, n4, n5, n6, n7):
    super().__init__()
    Conv1d = partial(nn.Conv1d,
                            stride=1, padding="same", bias=False
                            )
    DefaultMaxpool = partial(nn.MaxPool1d,
                             stride=1, padding=n7//2)
    self.column1 = Conv1d(in_channels = in_channels,
                                 out_channels = 16,
                                 kernel_size=n1)
    self.column2 = nn.Sequential(
        Conv1d(in_channels = in_channels,
                      out_channels = 16,
                      kernel_size=n5),
        nn.ReLU(),
        Conv1d(in_channels = 16,
                      out_channels = 16,
                      kernel_size=n2)
    )
    self.column3 = nn.Sequential(
        Conv1d(in_channels = in_channels,
                      out_channels = 16,
                      kernel_size=n6),
        nn.ReLU(),
        Conv1d(in_channels = 16,
                      out_channels = 16,
                      kernel_size=n3)
    )
    self.column4 = nn.Sequential(
        DefaultMaxpool(kernel_size=n7),
        Conv1d(in_channels = in_channels,
                      out_channels = 16,
                      kernel_size=n4)
    )
    self.output_dim = 64
    self.linear1 = nn.Linear(self.output_dim, 32)
    self.linear_output = nn.Linear(32, 24) #warning, see below to get to this layer we need to pool

  def forward(self, inputs):
    inputs = inputs.permute(0, 2, 1) #[batch, 100, 19] to [batch, 19, 100] 19 channels!

    out1 = self.column1(inputs)
    out2 = self.column2(inputs)
    out3 = self.column3(inputs)
    out4 = self.column4(inputs)

    combined = torch.cat((out1, out2, out3, out4), dim=1)

    inception_output = F.relu(combined)
    #inception_output.shape = (256=batch, 64, 100)

    pooled = inception_output.mean(dim=2)
    #pooled.shape = (256=batch, 64)

    pooled = self.linear1(pooled)
    #pooled.shape = (256=batch, 32)
    pooled = F.relu(pooled)

    return self.linear_output(pooled)

In [ ]:
torch.manual_seed(42)
model_inception2 = InceptionModule2(19, 1, 3, 5, 1, 1, 1, 3)
model_inception2.to(device)

n_epochs = 40
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model_inception2.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=24).to(device)

history = train(model_inception2, optimizer, xentropy, accuracy, train_loader, val_loader,
                n_epochs)

Epoch 1/40,                      train loss: 2.6048, train metric: 16.07%, valid metric: 20.98%
Epoch 2/40,                      train loss: 2.3856, train metric: 22.55%, valid metric: 23.74%
Epoch 3/40,                      train loss: 2.3120, train metric: 25.12%, valid metric: 26.15%
Epoch 4/40,                      train loss: 2.2628, train metric: 26.73%, valid metric: 27.20%
Epoch 5/40,                      train loss: 2.2306, train metric: 27.69%, valid metric: 27.31%
Epoch 6/40,                      train loss: 2.2007, train metric: 28.57%, valid metric: 28.94%
Epoch 7/40,                      train loss: 2.1753, train metric: 29.27%, valid metric: 29.47%
Epoch 8/40,                      train loss: 2.1493, train metric: 30.13%, valid metric: 29.59%
Epoch 9/40,                      train loss: 2.1214, train metric: 31.00%, valid metric: 31.48%
Epoch 10/40,                      train loss: 2.0931, train metric: 31.78%, valid metric: 32.23%
Epoch 11/40,                      train

## Inception Block
To control the architecture a bit better, we separate the inception block from the permutation, pooling and classifier

In [22]:
# increase the number of output channels from the single branches
# n1 n2 n3 n4
#    n5 n6 n7
# This is the module without permutation, pooling and classifier
# WARNING: this block makes [B, 19, 100] -> [B, 64, 100] so before use the data has to
# be permuted, this block cannot be used alone
# after the block apply x = x.mean(dim=2) to bring the dimension to [B, 64]
class InceptionBlock(nn.Module):
  def __init__(self, in_channels, n1, n2, n3, n4, n5, n6, n7):
    super().__init__()
    Conv1d = partial(nn.Conv1d,
                            stride=1, padding="same", bias=False
                            )
    DefaultMaxpool = partial(nn.MaxPool1d,
                             stride=1, padding=n7//2)
    self.column1 = Conv1d(in_channels = in_channels,
                                 out_channels = 16,
                                 kernel_size=n1)
    self.column2 = nn.Sequential(
        Conv1d(in_channels = in_channels,
                      out_channels = 16,
                      kernel_size=n5),
        nn.ReLU(),
        Conv1d(in_channels = 16,
                      out_channels = 16,
                      kernel_size=n2)
    )
    self.column3 = nn.Sequential(
        Conv1d(in_channels = in_channels,
                      out_channels = 16,
                      kernel_size=n6),
        nn.ReLU(),
        Conv1d(in_channels = 16,
                      out_channels = 16,
                      kernel_size=n3)
    )
    self.column4 = nn.Sequential(
        DefaultMaxpool(kernel_size=n7),
        Conv1d(in_channels = in_channels,
                      out_channels = 16,
                      kernel_size=n4)
    )


  def forward(self, inputs):

    out1 = self.column1(inputs)
    out2 = self.column2(inputs)
    out3 = self.column3(inputs)
    out4 = self.column4(inputs)

    combined = torch.cat((out1, out2, out3, out4), dim=1)

    return F.relu(combined)

## Stacked Inception Modules


In [ ]:
class InceptionModule3(nn.Module):
  def __init__(self, in_channels, n1, n2, n3, n4, n5, n6, n7):
    super().__init__()

    self.inception1 = InceptionBlock(
        in_channels, n1, n2, n3, n4, n5, n6, n7
    )

    self.inception2 = InceptionBlock(
        64, n1, n2, n3, n4, n5, n6, n7
    )

    self.linear1 = nn.Linear(64, 32)
    self.linear_output = nn.Linear(32, 24)

  def forward(self, inputs):
    inputs = inputs.permute(0, 2, 1)  # [B, 100, 19] -> [B, 19, 100]

    x = self.inception1(inputs)        # [B, 19, 100] -> [B, 64, 100]
    x = self.inception2(x)             # [B, 64, 100] -> [B, 64, 100]

    x = x.mean(dim=2)                  # [B, 64]
    x = self.linear1(x)                # [B, 32]
    x = F.relu(x)

    return self.linear_output(x)       # [B, 24]

In [ ]:
torch.manual_seed(42)
model_inception3 = InceptionModule3(19, 1, 3, 5, 1, 1, 1, 3)
model_inception3.to(device)

n_epochs = 40
early_stopping = 10
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model_inception3.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=24).to(device)

history = train(model_inception3, optimizer, xentropy, accuracy, train_loader, val_loader,
                n_epochs, early_stopping=early_stopping)

Epoch 1/40,                      train loss: 2.6433, train metric: 13.75%, valid metric: 19.85% Best!
Epoch 2/40,                      train loss: 2.3759, train metric: 22.77%, valid metric: 23.78% Best!
Epoch 3/40,                      train loss: 2.2981, train metric: 24.95%, valid metric: 26.06% Best!
Epoch 4/40,                      train loss: 2.2378, train metric: 26.89%, valid metric: 26.31% Best!
Epoch 5/40,                      train loss: 2.1764, train metric: 28.79%, valid metric: 28.76% Best!
Epoch 6/40,                      train loss: 2.1156, train metric: 30.67%, valid metric: 30.48% Best!
Epoch 7/40,                      train loss: 2.0576, train metric: 32.33%, valid metric: 32.48% Best!
Epoch 8/40,                      train loss: 2.0027, train metric: 34.11%, valid metric: 35.52% Best!
Epoch 9/40,                      train loss: 1.9542, train metric: 35.65%, valid metric: 37.02% Best!
Epoch 10/40,                      train loss: 1.9124, train metric: 36.95%, valid 

In [ ]:
n_epochs = 30
history2 = train(model_inception3, optimizer, xentropy, accuracy, train_loader, val_loader,
                n_epochs, early_stopping=early_stopping)

Epoch 1/30,                      train loss: 1.2598, train metric: 58.26%, valid metric: 57.24% Best!
Epoch 2/30,                      train loss: 1.2488, train metric: 58.59%, valid metric: 56.83%
Epoch 3/30,                      train loss: 1.2422, train metric: 58.69%, valid metric: 57.33% Best!
Epoch 4/30,                      train loss: 1.2311, train metric: 59.14%, valid metric: 58.19% Best!
Epoch 5/30,                      train loss: 1.2247, train metric: 59.33%, valid metric: 58.09%
Epoch 6/30,                      train loss: 1.2185, train metric: 59.64%, valid metric: 58.88% Best!
Epoch 7/30,                      train loss: 1.2085, train metric: 59.78%, valid metric: 58.56%
Epoch 8/30,                      train loss: 1.2028, train metric: 60.07%, valid metric: 58.08%
Epoch 9/30,                      train loss: 1.1974, train metric: 60.07%, valid metric: 57.59%
Epoch 10/30,                      train loss: 1.1653, train metric: 61.11%, valid metric: 59.40% Best!
Epoch 11/

In [ ]:
n_epochs = 30
history3 = train(model_inception3, optimizer, xentropy, accuracy, train_loader, val_loader,
                n_epochs, early_stopping=early_stopping)

Epoch 1/30,                      train loss: 1.0956, train metric: 63.42%, valid metric: 61.51% Best!
Epoch 2/30,                      train loss: 1.0948, train metric: 63.43%, valid metric: 61.59% Best!
Epoch 3/30,                      train loss: 1.0942, train metric: 63.38%, valid metric: 61.62% Best!
Epoch 4/30,                      train loss: 1.0942, train metric: 63.43%, valid metric: 61.62% Best!
Epoch 5/30,                      train loss: 1.0931, train metric: 63.45%, valid metric: 61.47%
Epoch 6/30,                      train loss: 1.0929, train metric: 63.47%, valid metric: 61.59%
Epoch 7/30,                      train loss: 1.0900, train metric: 63.58%, valid metric: 61.68% Best!
Epoch 8/30,                      train loss: 1.0895, train metric: 63.64%, valid metric: 61.81% Best!
Epoch 9/30,                      train loss: 1.0892, train metric: 63.56%, valid metric: 61.72%
Epoch 10/30,                      train loss: 1.0888, train metric: 63.54%, valid metric: 61.67%
Epo

In [ ]:
torch.save(
    model_inception3.state_dict(),
    "/content/drive/MyDrive/CFM_2024_Challenge/stacked_inception_1.pt"
)

# Bi-Gru Block

In [32]:
class GruBiBloc(nn.Module):
    def __init__(self, n_layers=2, input_dim=19,
                 hidden_dim=64, dropout=0.2):
        super().__init__()

        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=n_layers,
            batch_first=True, dropout=dropout, bidirectional=True)

    def forward(self, X):
      # X: [B, 100, 19]

      _outputs, hidden_states = self.gru(X)
      # _outputs: [B, 100, 128]
      # hidden_states: [4, B, 64]

      # concatenate forward and backward hidden states
      n_dims = 2*self.gru.hidden_size # n_dims = 128

      top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
      # top_states = hidden_states[-2:]
      # top_states: [2, B, 64] (extracts final layer)
      # top_states = top_states.permute(1, 0, 2)
      # top_states: [B, 2, 64]
      # top_states = top_states.reshape(-1, n_dims)
      # top_states: [B, 128] -> formula: [B, num_directions * hidden_dim]

      return top_states # [B, 128]

#BiGru+Inception

In [33]:
class BiGruInception(nn.Module):
  def __init__(self, in_channels, n1, n2, n3, n4, n5, n6, n7):
    super().__init__()

    self.inception1 = InceptionBlock(
        in_channels, n1, n2, n3, n4, n5, n6, n7
    )

    self.inception2 = InceptionBlock(
        64, n1, n2, n3, n4, n5, n6, n7
    )
    self.gru_block = GruBiBloc()

    self.linear_gru = nn.Linear(128, 64)
    self.linear_join1 = nn.Linear(128, 64)
    self.linear_join2 = nn.Linear(64, 32)
    self.linear_output = nn.Linear(32, 24)

  def forward(self, inputs): # inputs: [B,100, 19]
    p_inputs = inputs.permute(0, 2, 1)  # [B, 100, 19] -> [B, 19, 100]

    x1 = self.inception1(p_inputs)        # [B, 19, 100] -> [B, 64, 100]
    x1 = self.inception2(x1)             # [B, 64, 100] -> [B, 64, 100]

    x1 = x1.mean(dim=2)                  # [B, 64]

    x2 = self.gru_block(inputs) # [B, 100, 19] -> [B, 128]
    x2 = self.linear_gru(x2)       # [B, 128] -> [B, 64]

    combined = torch.cat((x1, x2), dim=1) # [B, 64]+[B, 64] ->[B, 128]
    x = F.relu(combined)

    x = self.linear_join1(x) # [B, 128] -> [B, 64]
    x = F.relu(x)

    x = self.linear_join2(x) # [B, 64] -> [B, 32]
    x = F.relu(x)

    return self.linear_output(x)       #[B,32] -> [B, 24]

## Training

In [34]:
torch.manual_seed(42)
model_BiGru_Inception1 = BiGruInception(19, 1, 3, 5, 1, 1, 1, 3)
model_BiGru_Inception1.to(device)

n_epochs = 50
early_stopping = 10
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model_BiGru_Inception1.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=24).to(device)

history = train(model_BiGru_Inception1, optimizer, xentropy, accuracy, train_loader, val_loader,
                n_epochs, early_stopping=early_stopping)

Epoch 1/50,                      train loss: 2.5474, train metric: 16.43%, valid metric: 22.91% Best!
Epoch 2/50,                      train loss: 2.1972, train metric: 27.64%, valid metric: 30.13% Best!
Epoch 3/50,                      train loss: 1.9634, train metric: 35.36%, valid metric: 36.99% Best!
Epoch 4/50,                      train loss: 1.8008, train metric: 40.37%, valid metric: 41.57% Best!
Epoch 5/50,                      train loss: 1.6738, train metric: 44.34%, valid metric: 45.40% Best!
Epoch 6/50,                      train loss: 1.5686, train metric: 47.47%, valid metric: 46.10% Best!
Epoch 7/50,                      train loss: 1.4741, train metric: 50.50%, valid metric: 50.90% Best!
Epoch 8/50,                      train loss: 1.3899, train metric: 53.22%, valid metric: 51.37% Best!
Epoch 9/50,                      train loss: 1.3184, train metric: 55.32%, valid metric: 55.03% Best!
Epoch 10/50,                      train loss: 1.2550, train metric: 57.48%, valid 

In [35]:
history2 = train(model_BiGru_Inception1, optimizer, xentropy, accuracy, train_loader, val_loader,
                n_epochs=50, early_stopping=early_stopping)

Epoch 1/50,                      train loss: 0.5263, train metric: 81.23%, valid metric: 69.11% Best!
Epoch 2/50,                      train loss: 0.5249, train metric: 81.34%, valid metric: 69.30% Best!
Epoch 3/50,                      train loss: 0.5217, train metric: 81.42%, valid metric: 69.16%
Epoch 4/50,                      train loss: 0.5207, train metric: 81.52%, valid metric: 69.16%
Epoch 5/50,                      train loss: 0.5172, train metric: 81.56%, valid metric: 69.20%
Epoch 6/50,                      train loss: 0.5092, train metric: 81.93%, valid metric: 69.18%
Epoch 7/50,                      train loss: 0.5082, train metric: 81.89%, valid metric: 69.27%
Epoch 8/50,                      train loss: 0.5053, train metric: 82.00%, valid metric: 69.16%
Epoch 9/50,                      train loss: 0.4996, train metric: 82.29%, valid metric: 69.37% Best!
Epoch 10/50,                      train loss: 0.4991, train metric: 82.35%, valid metric: 69.38% Best!
Epoch 11/50,   

In [36]:
torch.save(
    model_BiGru_Inception1.state_dict(),
    "/content/drive/MyDrive/CFM_2024_Challenge/BiGru_Inception1.pt"
)

# Testing

In [ ]:
test_path = Path("/content/drive/MyDrive/CFM_2024_Challenge/Xtest.parquet")

df_test = pd.read_parquet(test_path)

# get_dummies transforms the selected columns into one-hot
X_test = pd.get_dummies(
    df_test,
    columns=cat_columns,
    dtype=np.float32
)

del df_test
gc.collect()

# we remove obs_id and order_id for the moment
X_test.drop(columns=["obs_id", "order_id"], inplace=True)

In [ ]:
# scaling
means = X_test[num_cols].mean()
stds = X_test[num_cols].std()

X_test[num_cols] = (X_test[num_cols] - means) / stds

In [ ]:
X_test.shape

(8160000, 19)

In [ ]:
X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32)

del X_test
gc.collect()

22

In [ ]:
class StockTestDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X) // 100

    def __getitem__(self, index):
        start = index * 100
        end = start + 100
        return self.X[start:end]

In [ ]:
test_set = StockTestDataset(X_test_tensor)
del X_test_tensor
gc.collect()

test_loader = DataLoader(test_set,batch_size=64,shuffle=False)

In [ ]:
# put the model in evaluation mode (e.g. disable dropout)
model.eval()

# list where we will store all predicted stock classes
predictions = []

# no gradients for predictions
with torch.no_grad():

    # take one batch from the test set
    for X_batch in test_loader:

        # move the batch to the same device as the model
        X_batch = X_batch.to(device)

        # for each observation, the model outputs 24 logits
        logits = model(X_batch)

        # choose the class with the largest logit
        preds = logits.argmax(dim=1)

        # move predictions back to CPU and add them to our list
        predictions.extend(preds.cpu().numpy())

In [ ]:
submission = pd.DataFrame({
    "obs_id": np.arange(len(predictions)),
    "eqt_code_cat": predictions
})

submission_path = "/content/drive/MyDrive/CFM_2024_Challenge/predictions2.csv"

submission.to_csv(submission_path, index=False)


In [ ]:
submission.head()

,obs_id,eqt_code_cat
0,0,11
1,1,3
2,2,4
3,3,1
4,4,16


In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/CFM_2024_Challenge/gru_model_2_scaled.pt"
)